# NEMESIS-GNSS Quickstart

This notebook walks through the full NEMESIS pipeline:
1. Load and inspect your IQ dataset
2. Train a detector
3. Evaluate performance
4. Visualize results
5. Run inference on new files

In [ ]:
# Install if not already done
# !pip install nemesis-gnss[all]

## 1. Dataset

In [ ]:
from nemesis.data import IQDataset

# Point this at your dataset directory
DATA_PATH = "./my_dataset"   # <-- change this

dataset = IQDataset(DATA_PATH)
print(f"Total samples: {len(dataset)}")
print(f"Class distribution: {dataset.class_counts}")

## 2. Visualize a Scalogram

In [ ]:
from nemesis.viz import plot_scalogram

sample_path, label = dataset.samples[0]
from nemesis.data.dataset import CLASS_NAMES
plot_scalogram(sample_path, label=CLASS_NAMES[label], show=True)

## 3. Train

In [ ]:
from nemesis import Trainer

trainer = Trainer(
    data_path=DATA_PATH,
    output_dir="./checkpoints",
    config={
        "pretrain_epochs": 10,   # increase for better results
        "probe_epochs": 30,
        "batch_size": 16,
    }
)
trainer.fit()

## 4. Evaluate

In [ ]:
from nemesis.eval import evaluate_detector
from nemesis import NEMESISDetector

detector = NEMESISDetector.load("./checkpoints")
results = evaluate_detector(detector, DATA_PATH, verbose=True)

## 5. ROC Curves

In [ ]:
from nemesis.viz import plot_roc_curve

plot_roc_curve(results["y_true"], results["y_proba"])

## 6. Predict on a New File

In [ ]:
result = detector.predict("path/to/your/sample.bin")
print(f"Label     : {result['label']}")
print(f"Spoofed   : {result['spoofed']}")
print(f"Confidence: {result['confidence']:.2%}")
print("\nPer-class probabilities:")
for cls, prob in result['probabilities'].items():
    print(f"  {cls:<14}: {prob:.2%}")

## 7. Generate HTML Report

In [ ]:
from nemesis.eval import generate_report

report_path = generate_report(
    results,
    history=trainer._history,
    output_path="./nemesis_report.html"
)
print(f"Report saved: {report_path}")